# Approaches B and C: pretrained transformer (Google Colab)

**Run on Colab with a GPU:** Runtime → Change runtime type → **T4 GPU** → Run all. About 25–35 minutes.

| | Approach | What happens |
|---|---|---|
| **B** | Pretrained, no training | cardiffnlp/twitter-xlm-roberta-base-sentiment used as is. Multilingual XLM-R, already trained for 3-class sentiment on tweets in 8 languages (Arabic and English included). |
| **C** | Fine-tuned | The same checkpoint, fine-tuned on **our** training split. Best epoch picked on val macro-F1. |

Both use the **same** reviews_for_colab.csv (made by python -m src.data) and the same split column as the
TF-IDF baseline, and write predictions in the same format. All scoring is done locally by src/evaluate.py,
so every approach goes through identical evaluation code.

**Input:** no upload needed. The notebook rebuilds reviews_for_colab.csv from the two public sources with this
repo's own src/ code (embedded below), then checks its SHA-256 against the file built locally, so the split is
provably identical. If the check fails, it stops.
**Output:** transformer_outputs.zip (predictions + logs) downloads automatically → unzip into reports/predictions/.
The last cell (optional) saves the fine-tuned model to Google Drive (MyDrive/uneeq_sentiment/xlmr_finetuned) for the demo.

In [1]:
!pip -q install -U "transformers>=4.44,<5" "accelerate>=0.33" scikit-learn
import torch, transformers, sklearn, platform
print("torch", torch.__version__, "| transformers", transformers.__version__, "| python", platform.python_version())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 28.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
torch 2.11.0+cu130 | transformers 4.57.6 | python 3.13.15
GPU: Tesla T4


## Rebuild the dataset (same code, same seed) and verify it byte for byte

In [2]:
!mkdir -p src data/raw data/processed

In [3]:
%%writefile src/__init__.py

Writing src/__init__.py


In [4]:
%%writefile src/config.py
"""Paths, label mapping and run settings shared by every script."""
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
PREDICTIONS = REPORTS / "predictions"

ARABIC_FILE = RAW / "hard_unbalanced_reviews.txt"      # HARD, UTF-16 TSV
ENGLISH_FILE = RAW / "tripadvisor_hotel_reviews.csv"   # TripAdvisor 20k

SEED = 42
LABELS = ["negative", "neutral", "positive"]           # fixed order everywhere
LABEL2ID = {l: i for i, l in enumerate(LABELS)}

# Star rating -> sentiment. Same mapping for both languages.
RATING_TO_LABEL = {1: "negative", 2: "negative", 3: "neutral", 4: "positive", 5: "positive"}

# Arabic is sampled down to roughly the English size so neither language dominates
# training or the overall metric. The natural label mix of each language is kept.
ARABIC_SAMPLE_SIZE = 20_000
MIN_WORDS = 3            # drop reviews shorter than this after cleaning
SPLIT = (0.70, 0.15, 0.15)

# Booking.com fills the review title with a score word when the guest writes none
# ("استثنائي" = exceptional -> always 5 stars). That word *is* the label, so it is
# stripped before anything else (see notebooks/01_eda.ipynb, "label leakage").
BOOKING_SCORE_WORDS = [
    "ضعيف جداً", "ضعيف جدا", "ضعيف", "مخيب للأمل", "مقبول", "مرضي", "حسن",
    "جيد جداً", "جيد جدا", "جيد", "رائع", "ممتاز", "استثنائي",
]

Writing src/config.py


In [5]:
%%writefile src/clean.py
"""Text cleaning.

Two levels, on purpose:
- light_clean    : whitespace/URL tidy only. Fed to the transformers, whose tokenizers were
                   trained on raw text (normalising Arabic letters would hurt them).
- clean_for_tfidf: language-specific normalisation that shrinks the vocabulary for the
                   bag-of-words baseline (Arabic letter variants, diacritics, elongation).
"""
import re

from src.config import BOOKING_SCORE_WORDS

_URL = re.compile(r"https?://\S+|www\.\S+")
_WS = re.compile(r"\s+")
_ARABIC_CHAR = re.compile(r"[؀-ۿ]")
_DIACRITICS = re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۭ]")  # tashkeel + Quranic marks
_TATWEEL = re.compile("ـ")
_REPEAT = re.compile(r"(.)\1{2,}")  # "رااااائع" / "soooo" -> 2 copies

# Leading Booking.com score word: optional stray quote, the word, then ". "
_SCORE_PREFIX = re.compile(
    r'^\s*"?(' + "|".join(sorted(map(re.escape, BOOKING_SCORE_WORDS), key=len, reverse=True)) + r")\.\s*"
)


def detect_language(text: str) -> str:
    """'ar' if at least 30% of the letters are Arabic, else 'en'. Good enough for two languages."""
    letters = [c for c in str(text) if c.isalpha()]
    if not letters:
        return "en"
    arabic = sum(bool(_ARABIC_CHAR.match(c)) for c in letters)
    return "ar" if arabic / len(letters) >= 0.3 else "en"


def strip_booking_score_title(text: str) -> str:
    """Remove the auto-generated score word Booking.com puts in front of untitled reviews.

    User-written titles are wrapped in “…” and are kept; only the unquoted score word goes.
    """
    return _SCORE_PREFIX.sub("", str(text), count=1)


def light_clean(text: str) -> str:
    text = _URL.sub(" ", str(text))
    text = text.replace("“", '"').replace("”", '"')
    return _WS.sub(" ", text).strip(" .")


def normalize_arabic(text: str) -> str:
    text = _DIACRITICS.sub("", text)
    text = _TATWEEL.sub("", text)
    text = re.sub("[إأآٱ]", "ا", text)
    text = text.replace("ى", "ي").replace("ة", "ه").replace("ؤ", "و").replace("ئ", "ي")
    return text


def clean_for_tfidf(text: str, lang: str) -> str:
    text = light_clean(text)
    if lang == "ar":
        text = normalize_arabic(text)
    text = text.lower()
    text = _REPEAT.sub(r"\1\1", text)
    text = re.sub(r"[\"'“”«»()\[\]]", " ", text)
    return _WS.sub(" ", text).strip()

Writing src/clean.py


In [6]:
%%writefile src/data.py
"""Build the bilingual dataset and the one fixed split every approach is scored on.

    python -m src.data

Output: data/processed/reviews.parquet with columns
    id, lang, rating, label, text (light clean, for transformers), text_tfidf, split
and data/processed/split_summary.csv.
"""
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.clean import clean_for_tfidf, light_clean, strip_booking_score_title
from src.config import (ARABIC_FILE, ARABIC_SAMPLE_SIZE, ENGLISH_FILE, MIN_WORDS, PROCESSED,
                        RATING_TO_LABEL, SEED, SPLIT)


def load_arabic(path=ARABIC_FILE) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", encoding="utf-16", quoting=3)
    df = df.rename(columns={"review": "raw_text"})[["rating", "raw_text"]]
    df["raw_text"] = df["raw_text"].astype(str)
    stripped = df["raw_text"].map(strip_booking_score_title)
    df["had_score_title"] = stripped != df["raw_text"]
    df["text"] = stripped.map(light_clean)
    df["lang"] = "ar"
    return df


def load_english(path=ENGLISH_FILE) -> pd.DataFrame:
    df = pd.read_csv(path).rename(columns={"Review": "raw_text", "Rating": "rating"})
    df["raw_text"] = df["raw_text"].astype(str)
    df["had_score_title"] = False
    df["text"] = df["raw_text"].map(light_clean)
    df["lang"] = "en"
    return df


def tidy(df: pd.DataFrame) -> pd.DataFrame:
    """Map ratings to labels, drop very short and duplicate reviews (within a language)."""
    df = df[df["rating"].isin(RATING_TO_LABEL)].copy()
    df["rating"] = df["rating"].astype(int)
    df["label"] = df["rating"].map(RATING_TO_LABEL)
    df = df[df["text"].str.split().str.len() >= MIN_WORDS]
    # Same text with different ratings would be contradictory labels; keep the first.
    df = df.drop_duplicates(subset=["lang", "text"])
    return df


def sample_arabic(ar: pd.DataFrame, n=ARABIC_SAMPLE_SIZE) -> pd.DataFrame:
    """Stratified random sample that keeps the natural Arabic label mix."""
    keep, _ = train_test_split(ar.index, train_size=n, stratify=ar["label"], random_state=SEED)
    return ar.loc[keep].reset_index(drop=True)


def split(df: pd.DataFrame) -> pd.DataFrame:
    """70/15/15, stratified by language x label so every split has the same mix."""
    strat = df["lang"] + "_" + df["label"]
    train, rest = train_test_split(df.index, test_size=1 - SPLIT[0], stratify=strat, random_state=SEED)
    val, test = train_test_split(rest, test_size=SPLIT[2] / (SPLIT[1] + SPLIT[2]),
                                 stratify=strat.loc[rest], random_state=SEED)
    df["split"] = "train"
    df.loc[val, "split"] = "val"
    df.loc[test, "split"] = "test"
    return df


def build() -> pd.DataFrame:
    ar = tidy(load_arabic())
    en = tidy(load_english())
    ar = sample_arabic(ar)
    df = pd.concat([ar, en], ignore_index=True)
    df["text_tfidf"] = [clean_for_tfidf(t, l) for t, l in zip(df["text"], df["lang"])]
    df = split(df)
    df.insert(0, "id", [f"{l}_{i:05d}" for i, l in enumerate(df["lang"])])
    cols = ["id", "lang", "rating", "label", "text", "text_tfidf", "split", "had_score_title"]
    return df[cols]


if __name__ == "__main__":
    PROCESSED.mkdir(parents=True, exist_ok=True)
    df = build()
    df.to_parquet(PROCESSED / "reviews.parquet", index=False)
    # Plain CSV of the test/val/train text too: this is what gets uploaded to Colab.
    df[["id", "lang", "label", "text", "split"]].to_csv(PROCESSED / "reviews_for_colab.csv", index=False)
    summary = pd.crosstab([df["split"], df["lang"]], df["label"], margins=True)
    summary.to_csv(PROCESSED / "split_summary.csv")
    print(summary)
    print(f"\nSaved {len(df):,} reviews to {PROCESSED / 'reviews.parquet'}")

Writing src/data.py


In [7]:
import os, hashlib
os.makedirs("data/raw", exist_ok=True); os.makedirs("data/processed", exist_ok=True)
if not os.path.exists("data/raw/hard_unbalanced_reviews.txt"):
    !git clone -q --depth 1 https://github.com/elnagara/HARD-Arabic-Dataset.git /tmp/hard
    !(which unrar >/dev/null && unrar x -o+ -inul /tmp/hard/data/unbalanced-reviews.rar /tmp/hard/data/) || (apt-get -qq install -y unar >/dev/null && unar -q -f -o /tmp/hard/data /tmp/hard/data/unbalanced-reviews.rar)
    !mv /tmp/hard/data/unbalanced-reviews.txt data/raw/hard_unbalanced_reviews.txt
if not os.path.exists("data/raw/tripadvisor_hotel_reviews.csv"):
    !curl -sL -o data/raw/tripadvisor_hotel_reviews.csv https://raw.githubusercontent.com/YashSDholam/Tripadvisor-Hotel-Review-Sentiment-Analysis-using-LSTM-Neural-Network/main/tripadvisor_hotel_reviews.csv
!python -m src.data
got = hashlib.sha256(open("data/processed/reviews_for_colab.csv", "rb").read()).hexdigest()
EXPECTED = "7e11556c12d294a9bfdc6d03d4f961038ea222b96ccaef67d651ff513f41fdbc"
assert got == EXPECTED, f"Dataset differs from the local build ({got}). Stop: the comparison would not be fair."
print("Dataset identical to the local build:", got[:16])

label       negative  neutral  positive    All
split lang                                    
test  ar         392      595      2013   3000
      en         482      328      2264   3074
train ar        1830     2780      9389  13999
      en        2250     1529     10565  14344
val   ar         393      596      2012   3001
      en         482      327      2264   3073
All             5829     6155     28507  40491

Saved 40,491 reviews to /content/data/processed/reviews.parquet
Dataset identical to the local build: 7e11556c12d294a9


In [8]:
import os, json, time, random
import numpy as np, pandas as pd
from google.colab import files

df = pd.read_csv("data/processed/reviews_for_colab.csv")
print(df.shape); print(pd.crosstab(df["split"], df["lang"]))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
LABELS = ["negative", "neutral", "positive"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
CHECKPOINT = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
MAX_LEN = 256          # covers ~95% of reviews; longer ones are truncated (documented limitation)
OUT = "outputs"; os.makedirs(OUT, exist_ok=True)
train_df, val_df, test_df = (df[df["split"] == s].reset_index(drop=True) for s in ("train", "val", "test"))

(40491, 5)
lang      ar     en
split              
test    3000   3074
train  13999  14344
val     3001   3073


In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained(CHECKPOINT)

# Truncation check: share of reviews longer than MAX_LEN tokens, per language.
lens = df["text"].map(lambda t: len(tok(t)["input_ids"]))
print(df.assign(n_tokens=lens).groupby("lang")["n_tokens"].describe(percentiles=[.5, .9, .95]))
print("Truncated share:", (lens > MAX_LEN).groupby(df["lang"]).mean().round(3).to_dict())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

        count        mean         std   min    50%    90%    95%     max
lang                                                                    
ar    20000.0   41.818450   39.174020   5.0   30.0   84.0  110.0   820.0
en    20491.0  159.566883  156.802296  12.0  117.0  307.0  421.0  3338.0
Truncated share: {'ar': 0.003, 'en': 0.144}


In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def checkpoint_label_order(model):
    # Map the checkpoint's own label ids to our fixed order (negative, neutral, positive).
    id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
    if set(id2label.values()) != set(LABELS):   # e.g. LABEL_0/1/2 -> cardiffnlp order is neg/neu/pos
        id2label = {0: "negative", 1: "neutral", 2: "positive"}
    return [ [k for k, v in id2label.items() if v == l][0] for l in LABELS ]

@torch.no_grad()
def predict(model, texts, batch_size=64):
    model.eval().to(device)
    order = checkpoint_label_order(model)
    # Sort by length so batches pad less (much faster); restore order afterwards.
    idx = np.argsort([len(t) for t in texts])
    probs = np.zeros((len(texts), 3), dtype=np.float32)
    t0 = time.time()
    for s in range(0, len(texts), batch_size):
        b = idx[s:s + batch_size]
        enc = tok([texts[i] for i in b], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(**enc).logits.float()
        probs[b] = torch.softmax(logits, -1)[:, order].cpu().numpy()
    if device == "cuda": torch.cuda.synchronize()
    return probs, (time.time() - t0) / len(texts) * 1000

def save_preds(name, part, frame):
    probs, ms = predict(model, frame["text"].tolist())
    out = pd.DataFrame(probs, columns=[f"p_{l}" for l in LABELS])
    out.insert(0, "pred", np.array(LABELS)[probs.argmax(1)])
    out.insert(0, "id", frame["id"].values)
    out.to_csv(f"{OUT}/{name}_{part}.csv", index=False)
    from sklearn.metrics import f1_score
    print(f"{name} {part}: macro-F1 {f1_score(frame['label'], out['pred'], average='macro'):.4f} | {ms:.2f} ms/review ({device} batch)")
    return ms

## Approach B: pretrained model, no training

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT)
print(model.config.id2label)
lat = {}
for part, frame in (("val", val_df), ("test", test_df)):
    lat[part] = save_preds("zeroshot_xlmr", part, frame)
json.dump({"ms_per_review_batch_gpu": round(lat["test"], 3), "hardware": torch.cuda.get_device_name(0) if device == "cuda" else "cpu"},
          open(f"{OUT}/zeroshot_xlmr_latency.json", "w"))

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

{0: 'negative', 1: 'neutral', 2: 'positive'}


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

zeroshot_xlmr val: macro-F1 0.5270 | 3.46 ms/review (cuda batch)
zeroshot_xlmr test: macro-F1 0.5173 | 2.33 ms/review (cuda batch)


## Approach C: fine-tune on our training split
- Starts from the same checkpoint, so the difference between B and C is purely **our data**.
- Class-weighted cross-entropy (n / (3 * n_class)): the same weighting the TF-IDF baseline picked on val.
- 3 epochs max, learning rate 2e-5, warmup 10%, best epoch by **val macro-F1**. The test split is never seen during training or selection.

In [12]:
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
from sklearn.metrics import f1_score

class ReviewDS(Dataset):
    def __init__(self, frame):
        self.enc = tok(frame["text"].tolist(), truncation=True, max_length=MAX_LEN)
        self.y = frame["label"].map(LABEL2ID).tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}; item["labels"] = self.y[i]; return item

model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=3)
# Make the head's label ids match our order explicitly.
order = checkpoint_label_order(model)
assert order == [0, 1, 2], f"unexpected label order {order}"
model.config.id2label = {i: l for i, l in enumerate(LABELS)}; model.config.label2id = LABEL2ID

counts = train_df["label"].value_counts().reindex(LABELS).values
class_w = torch.tensor(len(train_df) / (3 * counts), dtype=torch.float32)
print("class weights", dict(zip(LABELS, class_w.round(decimals=3).tolist())))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=class_w.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def metrics(p):
    pred = p.predictions.argmax(-1)
    return {"macro_f1": f1_score(p.label_ids, pred, average="macro")}

args = TrainingArguments(
    output_dir="ckpt", num_train_epochs=3, learning_rate=2e-5, warmup_ratio=0.1, weight_decay=0.01,
    per_device_train_batch_size=32, per_device_eval_batch_size=64, fp16=(device == "cuda"),
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, load_best_model_at_end=True,
    metric_for_best_model="macro_f1", greater_is_better=True, logging_steps=100, report_to="none",
    seed=SEED, group_by_length=True,
)
trainer = WeightedTrainer(model=model, args=args, train_dataset=ReviewDS(train_df), eval_dataset=ReviewDS(val_df),
                          data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics,
                          callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
t0 = time.time(); trainer.train(); train_minutes = (time.time() - t0) / 60
print(f"Training took {train_minutes:.1f} min")
json.dump({"log_history": trainer.state.log_history, "train_minutes": round(train_minutes, 1),
           "best_val_macro_f1": trainer.state.best_metric, "args": {k: str(v) for k, v in args.to_dict().items()
           if k in ("num_train_epochs", "learning_rate", "warmup_ratio", "weight_decay", "per_device_train_batch_size", "seed")},
           "max_len": MAX_LEN, "class_weights": class_w.tolist(), "checkpoint": CHECKPOINT},
          open(f"{OUT}/finetuned_xlmr_training_log.json", "w"), indent=2)

class weights {'negative': 2.315999984741211, 'neutral': 2.193000078201294, 'positive': 0.4729999899864197}


Epoch,Training Loss,Validation Loss,Macro F1
1,0.673000,0.707702,0.613858
2,0.566000,0.617803,0.715573
3,0.473400,0.639883,0.718911


Training took 15.1 min


In [13]:
model = trainer.model
lat = {}
for part, frame in (("val", val_df), ("test", test_df)):
    lat[part] = save_preds("finetuned_xlmr", part, frame)
json.dump({"ms_per_review_batch_gpu": round(lat["test"], 3), "hardware": torch.cuda.get_device_name(0) if device == "cuda" else "cpu"},
          open(f"{OUT}/finetuned_xlmr_latency.json", "w"))

finetuned_xlmr val: macro-F1 0.7186 | 1.87 ms/review (cuda batch)
finetuned_xlmr test: macro-F1 0.6968 | 1.91 ms/review (cuda batch)


## Download the predictions (small zip, for reports/predictions/)

In [14]:
!cd outputs && zip -q ../transformer_outputs.zip *
files.download("transformer_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Optional: save the fine-tuned model to Google Drive (≈1.1 GB, only needed for the demo's transformer mode)

In [15]:
from google.colab import drive
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/uneeq_sentiment/xlmr_finetuned"
trainer.save_model(SAVE); tok.save_pretrained(SAVE)
print("Model saved to", SAVE, "-> download that folder into models/xlmr_finetuned/ for the demo")

Mounted at /content/drive
Model saved to /content/drive/MyDrive/uneeq_sentiment/xlmr_finetuned -> download that folder into models/xlmr_finetuned/ for the demo
